In [1]:
import os
import rootutils

rootutils.setup_root(os.getcwd(), indicator=".project-root", pythonpath=True)

from torchvision import transforms
import torch
from src.data.components.graphs_datamodules import (
    IMCBaseDictTransform,
    PickleDataset,
    PatchAugmentations,
)
from src.data.imc_datamodule import add_channel
import src.data.components.graphs_datamodules as gd
from torchvision.datasets import MNIST
from torch.utils.data import ConcatDataset, Dataset, random_split

In [2]:
from pathlib import Path

In [4]:
from pathlib import Path
import pickle

path = Path("/raid/tnocon/data") / "IMC" / "nsclc2_panel1_test.pkl"

with path.open("rb") as f:
    data = pickle.load(f)

In [14]:
list(data.keys())[0]

('/raid/immucan/immuvis_splits/test/nsclc2-panel1/imgs/full.immucan-p1.nsclc2.4080.tiff',)

In [22]:
data[('/raid/immucan/immuvis_splits/test/nsclc2-panel1/imgs/full.immucan-p1.nsclc2.4080.tiff',)][0]['position']

(slice(0, 113, None), slice(0, 113, None))

In [23]:
data[('/raid/immucan/immuvis_splits/test/nsclc2-panel1/imgs/full.immucan-p1.nsclc2.4080.tiff',)][1]['position']

(slice(0, 113, None), slice(113, 226, None))

In [24]:
data[('/raid/immucan/immuvis_splits/test/nsclc2-panel1/imgs/full.immucan-p1.nsclc2.4080.tiff',)][2]

{'r0_nf': {'embedding': array([[[[ 0.0662582 ,  0.20338011,  0.04152491, ..., -0.07691472,
            -0.02757137,  0.03017415],
           [-0.10378787, -0.02558527, -0.18015344, ..., -0.10153525,
            -0.02897811,  0.17104399],
           [ 0.01002203,  0.12697864,  0.51601326, ..., -0.01360824,
             0.40437213,  0.34285927],
           ...,
           [ 0.05496953, -0.15617368, -0.06874275, ..., -0.06576315,
            -0.26145387, -0.06584876],
           [-0.05823384, -0.16882597, -0.3950214 , ..., -0.02333965,
            -0.25566608,  0.05054987],
           [-0.08063725, -0.07518306,  0.25336602, ...,  0.3221363 ,
            -0.13468242, -0.18369415]],
  
          [[-0.02675474, -0.05656582,  0.1307862 , ...,  0.15624966,
            -0.13863422, -0.08006217],
           [-0.02198555,  0.6172588 ,  0.28511596, ...,  0.59683144,
             0.09066576, -0.14419875],
           [ 0.08314537,  0.09530422, -0.11187308, ...,  0.5226181 ,
             0.14374354, 

In [28]:
for el in data[('/raid/immucan/immuvis_splits/test/nsclc2-panel1/imgs/full.immucan-p1.nsclc2.4080.tiff',)]:
    print(el['position'])

(slice(0, 113, None), slice(0, 113, None))
(slice(0, 113, None), slice(113, 226, None))
(slice(0, 113, None), slice(226, 339, None))
(slice(0, 113, None), slice(339, 452, None))
(slice(0, 113, None), slice(452, 565, None))
(slice(0, 113, None), slice(487, 600, None))
(slice(113, 226, None), slice(0, 113, None))
(slice(113, 226, None), slice(113, 226, None))
(slice(113, 226, None), slice(226, 339, None))
(slice(113, 226, None), slice(339, 452, None))
(slice(113, 226, None), slice(452, 565, None))
(slice(113, 226, None), slice(487, 600, None))
(slice(226, 339, None), slice(0, 113, None))
(slice(226, 339, None), slice(113, 226, None))
(slice(226, 339, None), slice(226, 339, None))
(slice(226, 339, None), slice(339, 452, None))
(slice(226, 339, None), slice(452, 565, None))
(slice(226, 339, None), slice(487, 600, None))
(slice(339, 452, None), slice(0, 113, None))
(slice(339, 452, None), slice(113, 226, None))
(slice(339, 452, None), slice(226, 339, None))
(slice(339, 452, None), slice(339

In [4]:
print(data[0]['img_path'])
print(data[36]['img_path'])
print(data[72]['img_path'])
print(data[108]['img_path'])
print(data[14]['img_path'])

/raid/immucan/IMC/NSCLC2/IMC1/unzipped/test/LUNG-NSCLC2-0715-FIXT-01-IMC1-01_#_IMMUcan_panel_1_1.10_#_THOR_#_f3aa2d13758c78b549284d40c20e65a0/img/IMMUcan_Batch20210701_LUNG_10018884-LUNG-VAR-TIS-01-IMC-01_001.tiff
/raid/immucan/IMC/NSCLC2/IMC1/unzipped/test/LUNG-NSCLC2-0715-FIXT-01-IMC1-01_#_IMMUcan_panel_1_1.10_#_THOR_#_f3aa2d13758c78b549284d40c20e65a0/img/IMMUcan_Batch20210701_LUNG_10018884-LUNG-VAR-TIS-01-IMC-01_002.tiff
/raid/immucan/IMC/NSCLC2/IMC1/unzipped/test/LUNG-NSCLC2-0715-FIXT-01-IMC1-01_#_IMMUcan_panel_1_1.10_#_THOR_#_f3aa2d13758c78b549284d40c20e65a0/img/IMMUcan_Batch20210701_LUNG_10018884-LUNG-VAR-TIS-01-IMC-01_003.tiff
/raid/immucan/IMC/NSCLC2/IMC1/unzipped/test/LUNG-NSCLC2-0715-FIXT-01-IMC1-01_#_IMMUcan_panel_1_1.10_#_THOR_#_f3aa2d13758c78b549284d40c20e65a0/img/IMMUcan_Batch20210701_LUNG_10018884-LUNG-VAR-TIS-01-IMC-01_004.tiff
/raid/immucan/IMC/NSCLC2/IMC1/unzipped/test/LUNG-NSCLC2-0715-FIXT-01-IMC1-01_#_IMMUcan_panel_1_1.10_#_THOR_#_f3aa2d13758c78b549284d40c20e65a0/im

In [5]:
res = {}
for i, el in enumerate(data):
    for k, v in el.items():
        if k == 'img_path':
            img_path = v
            if img_path not in res:
                res[img_path] = [i]
            

In [ ]:
train_path = Path("/raid/tnocon/data") / 'IMC' / 'train.pkl'
# make sure the directory exists
train_path.parent.mkdir(parents=True, exist_ok=True)

# save the object
with train_path.open("wb") as f:
    pickle.dump(data[:36 * 100], f)

In [8]:
test_path = Path("/raid/tnocon/data") / 'IMC' / 'test.pkl'
# make sure the directory exists
test_path.parent.mkdir(parents=True, exist_ok=True)

# save the object
with test_path.open("wb") as f:
    pickle.dump(data[36 * 100:], f)

In [4]:
from src.models.components.plot import restore_tensor
import matplotlib.pyplot as plt

base_transforms = IMCBaseDictTransform()

aug_transforms_train = gd.PatchAugmentations(
    prob=1.0,
    size=13,
    patch_size=1,
)

aug_transforms_val = gd.PatchAugmentations(
    prob=1.0,
    size=13,
    patch_size=1,
    is_validation=True,
)

dual_transforms_train = gd.DualOutputTransform(base_transforms, aug_transforms_train)

dual_transforms_val = gd.DualOutputTransform(base_transforms, aug_transforms_val)

train_path = Path("/raid/tnocon/data") / 'IMC' / 'train.h5'
test_path = Path("/raid/tnocon/data") / 'IMC' / 'test.h5'
trainset = PickleDataset(train_path, transform=dual_transforms_train)
testset = PickleDataset(train_path, transform=dual_transforms_val)
train_ratio, val_ratio, test_ratio, leftover_ratio = [3600, 1044, 0, 0]
size_testset = len(testset)
size_trainset = len(trainset)
data_train, _ = random_split(
    dataset=trainset,
    lengths=[train_ratio, size_trainset - train_ratio],
    generator=torch.Generator().manual_seed(42),
)
# dataset = ConcatDataset(datasets=[trainset, testset])
data_val, data_test, _ = random_split(
    dataset=testset,
    lengths=[val_ratio, test_ratio, size_testset - val_ratio - test_ratio],
    generator=torch.Generator().manual_seed(42),
)

train_dataset = gd.GridGraphDataset(grid_size=13, dataset=data_train, channels=[0])

train_loader = gd.DenseGraphDataLoader(
    dataset=train_dataset,
    batch_size=8,
    num_workers=7,
    pin_memory=False,
    persistent_workers=7 > 0,
)

In [16]:
trainset[0][0].shape

torch.Size([8, 169, 512])

In [6]:
for el in train_loader:
    break

In [11]:
vars(el).keys()

dict_keys(['node_features', 'edge_features', 'mask', 'argsort_augmented_features', 'perms', 'properties', 'y'])

In [12]:
el.argsort_augmented_features.shape

torch.Size([64, 169])

In [14]:
el.mask.shape

torch.Size([64, 169])

In [15]:

nodes_true = el.node_features
nodes_pred = el.node_features

# Compute the node-based loss
signal_power = torch.mean(nodes_true ** 2) 
noise_power = torch.mean((nodes_pred - nodes_true) ** 2)

# loss = 10 * torch.log10(signal_power / noise_power)

In [63]:
import h5py
import numpy as np

# One-time conversion from your pickle to HDF5
def convert_pickle_to_hdf5(pickle_path, hdf5_path):
    with open(pickle_path, 'rb') as f:
        data_list = pickle.load(f)
    
    with h5py.File(hdf5_path, 'w') as f:
        # Store each dictionary field as separate datasets
        for key in data_list[0].keys():
            if key == 'img_path':
                continue
            # Convert to numpy, keep float32, stack along first axis
            values = [item[key] for item in data_list]  # already float32
            f.create_dataset(key, data=np.stack(values, axis=0), dtype='float32')

In [ ]:
train_path = Path("/raid/tnocon/data") / 'IMC' / 'train.pkl'
train_hdf5_path = Path("/raid/tnocon/data") / 'IMC' / 'train.h5'

test_path = Path("/raid/tnocon/data") / 'IMC' / 'test.pkl'
test_hdf5_path = Path("/raid/tnocon/data") / 'IMC' / 'test.h5'

In [65]:
convert_pickle_to_hdf5(train_path, train_hdf5_path)

In [66]:
convert_pickle_to_hdf5(test_path, test_hdf5_path)

In [67]:
test_path = Path("/raid/tnocon/data") / 'IMC' / 'test.pkl'

with open(test_path, 'rb') as f:
    data_list = pickle.load(f)

In [68]:
data_list[0]['r0_nf']

array([[[-0.31997058, -0.11488403,  0.04572535, ..., -0.7873637 ,
         -0.52338105, -0.48128554],
        [-0.21776313, -0.25680414, -0.14512348, ..., -0.24081172,
         -0.01628412, -0.3098817 ],
        [ 0.01115622,  0.07891098, -0.1971682 , ..., -0.29762226,
          0.28093928, -0.5532867 ],
        ...,
        [-0.54629934, -0.08667167,  0.07792269, ..., -0.04829525,
          0.08995356, -0.20581272],
        [-0.47225195, -0.40551075, -0.23589738, ..., -0.23275362,
         -0.21130247, -0.29297823],
        [-0.6544423 , -0.39127392, -0.33342323, ..., -0.14733388,
         -0.30317014, -0.33758003]],

       [[-0.15339708, -0.04065932,  0.18081775, ..., -0.34217477,
         -0.12556429, -0.05842322],
        [-0.1224561 ,  0.45538798,  0.08024752, ...,  0.13663732,
          0.32097647, -0.1662379 ],
        [ 0.3859747 ,  0.395517  ,  0.13773231, ...,  0.09259814,
          0.0527259 , -0.0937293 ],
        ...,
        [-0.20801239,  0.6781688 ,  0.04099444, ...,  

In [69]:
import h5py

test_hdf5_path = Path("/raid/tnocon/data") / 'IMC' / 'test.h5'

with h5py.File(test_hdf5_path, 'r') as f:
    # Load only the item at index idx
    item = {key: f[key][0] for key in f.keys()}

In [73]:
item['r0_nf']

array([[[-0.31997058, -0.11488403,  0.04572535, ..., -0.7873637 ,
         -0.52338105, -0.48128554],
        [-0.21776313, -0.25680414, -0.14512348, ..., -0.24081172,
         -0.01628412, -0.3098817 ],
        [ 0.01115622,  0.07891098, -0.1971682 , ..., -0.29762226,
          0.28093928, -0.5532867 ],
        ...,
        [-0.54629934, -0.08667167,  0.07792269, ..., -0.04829525,
          0.08995356, -0.20581272],
        [-0.47225195, -0.40551075, -0.23589738, ..., -0.23275362,
         -0.21130247, -0.29297823],
        [-0.6544423 , -0.39127392, -0.33342323, ..., -0.14733388,
         -0.30317014, -0.33758003]],

       [[-0.15339708, -0.04065932,  0.18081775, ..., -0.34217477,
         -0.12556429, -0.05842322],
        [-0.1224561 ,  0.45538798,  0.08024752, ...,  0.13663732,
          0.32097647, -0.1662379 ],
        [ 0.3859747 ,  0.395517  ,  0.13773231, ...,  0.09259814,
          0.0527259 , -0.0937293 ],
        ...,
        [-0.20801239,  0.6781688 ,  0.04099444, ...,  